In [2]:
import sqlalchemy
from sqlalchemy import text
import pandas as pd

In [3]:
from dotenv import load_dotenv
import os
load_dotenv("db.env", override=True)
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_PORT = os.getenv("DB_PORT")

print(DB_USER, DB_PASS[:1], DB_HOST, DB_NAME, DB_PORT)

postgres 1 localhost db_benford 5432


In [4]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME)

engine = create_engine(url, connect_args={"connect_timeout": 10})
con = engine.connect()

In [ ]:
def select_query(connection, query: str) -> pd.DataFrame:
    df = pd.read_sql(text(query), connection)
    return df

select_query(con, "SELECT * FROM tb_gastos_deputados LIMIT 3")

Executando query:
SELECT * FROM tb_gastos_deputados LIMIT 3


,id,txnomeparlamentar,cpf,idecadastro,nucarteiraparlamentar,nulegislatura,sguf,sgpartido,codlegislatura,numsubcota,...,numparcela,txtpassageiro,txttrecho,numlote,numressarcimento,datpagamentorestituicao,vlrrestituicao,nudeputadoid,idedocumento,urldocumento
0,1,LID.GOV-CD,None,None,None,2023,None,None,57,1,...,0,None,None,2115566,None,None,None,2812,7877589,https://www.camara.leg.br/cota-parlamentar/doc...
1,2,LID.GOV-CD,None,None,None,2023,None,None,57,1,...,0,None,None,2123760,None,None,None,2812,7892756,https://www.camara.leg.br/cota-parlamentar/doc...
2,3,LID.GOV-CD,None,None,None,2023,None,None,57,1,...,0,None,None,2131750,None,None,None,2812,7907982,https://www.camara.leg.br/cota-parlamentar/doc...


In [10]:
select_query(con, "SELECT * FROM tb_clientes")

Executando query:
SELECT * FROM tb_clientes


,id,nome,usuario,email,senha_hash,is_admin,is_active,data_cadastro,deletado


In [20]:
!uv add bcrypt
import bcrypt
df_cliente_inserir = pd.DataFrame(data=
    {
        "nome": "Prandiano",
        "usuario": "prandiano",
        "email": "prandiano@exemplo.com",
        "senha_hash": bcrypt.hashpw("123".encode('utf-8'), bcrypt.gensalt()).decode('utf-8'),
        "is_admin": True
    }, index=[0]
)
df_cliente_inserir

Resolved 74 packages in 3ms
Checked 72 packages in 1ms


,nome,usuario,email,senha_hash,is_admin
0,Prandiano,prandiano,prandiano@exemplo.com,$2b$12$0oUYRzxu6Ptyszqbt9T6COu.VLwQgtK5hBSWEco...,True


In [21]:
!uv add bcrypt

def inserir_dados(connection, table_name: str, data: pd.DataFrame):
    data.to_sql(table_name, connection, if_exists='append', 
                index=False)

inserir_dados(con, "tb_clientes", df_cliente_inserir)

Resolved 74 packages in 3ms
Checked 72 packages in 1ms


In [22]:
select_query(con, "SELECT * FROM tb_clientes")

Executando query:
SELECT * FROM tb_clientes


,id,nome,usuario,email,senha_hash,is_admin,is_active,data_cadastro,deletado
0,1,Prandiano,prandiano,prandiano@exemplo.com,$2b$12$0oUYRzxu6Ptyszqbt9T6COu.VLwQgtK5hBSWEco...,True,True,2026-10-31 20:41:00.521245,False


In [ ]:
def deletar_usuario(connection, nome_usuario: str) -> int:
    query = text(f"""UPDATE tb_clientes 
                     SET deletado = TRUE 
                     WHERE usuario = '{nome_usuario}'
                """)
    resultado = connection.execute(query)
    connection.commit()
    return resultado.rowcount

deletar_usuario(con, "prandiano")
select_query(con, "SELECT * FROM tb_clientes")

Executando query:
SELECT * FROM tb_clientes


,id,nome,usuario,email,senha_hash,is_admin,is_active,data_cadastro,deletado
0,1,Prandiano,prandiano,prandiano@exemplo.com,$2b$12$0oUYRzxu6Ptyszqbt9T6COu.VLwQgtK5hBSWEco...,True,True,2026-10-31 20:41:00.521245,True


In [6]:
def buscar_usuario(con, nome_usuario: str) -> dict | None:
    query = text("""
        SELECT nome, usuario, senha_hash, is_admin
        FROM tb_clientes
        WHERE usuario = :usuario
            AND is_active = TRUE
            AND deletado = FALSE
    """)

    resultado = con.execute(query, {"usuario": nome_usuario})
    linha = resultado.mappings().first()
    return dict(linha) if linha else None

buscar_usuario(con, "prandiano")

{'nome': 'Prandiano',
 'usuario': 'prandiano',
 'senha_hash': '$2b$12$0oUYRzxu6Ptyszqbt9T6COu.VLwQgtK5hBSWEcoClEWopi9kwOY4C',
 'is_admin': True}